In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
import time
import re
from turtle import title

In [2]:
url = "https://www.jumia.ug/laptops/"
headers = {"User-Agent": "Mozilla/5.0"}

In [3]:

def get_page(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")
    return soup

pages=50
all_products = []

for page in range(1, pages+1):
    url = f"https://www.jumia.ug/laptops/?page={page}#catalog-listing"
    soup = get_page(url)
    products = soup.find_all("article", class_="prd _fb col c-prd")

    for product in products:
        name = product.find('h3', class_='name')
        name = name.text if name else None

        price = product.find('div', class_='prc')
        price = price.text if price else None

        old_price = product.find('div', class_='old')
        old_price = old_price.text if old_price else None

        discount = product.find('div', class_='bdg _dsct _sm')
        discount = discount.text if discount else None

        product_data = {'name':name, 'price':price, 'old_price':old_price, 'discount':discount}

        all_products.append(product_data)
    time.sleep(1)

df = pd.DataFrame(all_products)
df.to_csv('jumia_products.csv', index=False)

In [4]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   name       2000 non-null   str  
 1   price      2000 non-null   str  
 2   old_price  1898 non-null   str  
 3   discount   1898 non-null   str  
dtypes: str(4)
memory usage: 62.6 KB


,name,price,old_price,discount
0,Refurbished HP Probook 640 Core I5 8GB Ram 500...,"UGX 559,000","UGX 1,400,000",60%
1,Hp Refurbished Probook TouchScreen X360 Intel ...,"UGX 440,000","UGX 980,019",55%
2,Latitude 11'' Inch 3180/3190 4GB RAM 128GB SSD...,"UGX 349,000","UGX 700,000",50%
3,"11e Mini laptop,11.6"" Inch 4GB RAM,128GB SSD, ...","UGX 329,000","UGX 800,000",59%
4,"Travelmate 12 Inch Laptop 4GB RAM 128GB SSD, I...","UGX 356,400","UGX 800,000",55%


In [5]:
def data_cleaning(df):
    
    # Convert 'price' and 'discount' columns from boolean to string values
    df['price'] = df['price'].astype(str)
    df['discount'] = df['discount'].astype(str)

    # Removing data contains '-' from 'price' and 'discount' columns
    df = df[~df['price'].str.contains('-', na=False)]
    df = df[~df['discount'].str.contains('-', na=False)]
    
    # Removing 'UGX', ',' and '%' from 'price', 'old_price' and 'discount' columns
    df['price'] = df['price'].str.replace('UGX', '').str.replace(',', '').str.strip()
    df['old_price'] = df['old_price'].str.replace('UGX', '').str.replace(',', '').str.strip()
    
    # Convert 'price' and 'old_price' columns to numeric values
    df['price'] = df['price'].astype(float)
    df['old_price'] = df['old_price'].astype(float)

    return df

In [6]:
# Apply the data cleaning function to the new variable called data_jumia

data_jumia = pd.DataFrame(data_cleaning(df))
data_jumia.info()
data_jumia.head()

<class 'pandas.DataFrame'>
Index: 1994 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       1994 non-null   str    
 1   price      1994 non-null   float64
 2   old_price  1892 non-null   float64
 3   discount   1892 non-null   str    
dtypes: float64(2), str(2)
memory usage: 77.9 KB


,name,price,old_price,discount
0,Refurbished HP Probook 640 Core I5 8GB Ram 500...,559000.0,1400000.0,60%
1,Hp Refurbished Probook TouchScreen X360 Intel ...,440000.0,980019.0,55%
2,Latitude 11'' Inch 3180/3190 4GB RAM 128GB SSD...,349000.0,700000.0,50%
3,"11e Mini laptop,11.6"" Inch 4GB RAM,128GB SSD, ...",329000.0,800000.0,59%
4,"Travelmate 12 Inch Laptop 4GB RAM 128GB SSD, I...",356400.0,800000.0,55%


In [7]:
data_jumia[['price', 'old_price']].describe().round()

,price,old_price
count,1994.0,1892.0
mean,866373.0,1507911.0
std,888440.0,2723604.0
min,70000.0,120000.0
25%,495500.0,899000.0
50%,600000.0,1150000.0
75%,853750.0,1589990.0
max,18500000.0,100000000.0


In [31]:
# Extracting RAM information from the 'name' column
def extract_ram(name):
    if not isinstance(name, str):
        return None
    
    match = re.search(r'(\d+)\s?GB\s?RAM', name, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None


# Extracting (storage information) from the 'name' column
def extract_storage(name):
    if not isinstance(name, str):
        return None
    
    match = re.search(r'(\d+)\s?(GB|TB)', name, re.IGNORECASE)
    if match:
        size = int(match.group(1))
        unit = match.group(2).upper()
        
        if unit == "TB":
            size *= 1000  # convert to GB
            
        return size
    return None


# Extracting storage type (SSD or HDD) from the 'name' column
def extract_storage_type(name):
    if not isinstance(name, str):
        return None
    
    if "ssd" in name.lower():
        return "SSD"
    elif "hdd" in name.lower():
        return "HDD"
    
    return None
'''
# Extracting (CPU) information from the 'name' column
def extract_cpu(name):
    if not isinstance(name, str):
        return None
    
    match = re.search(r'(Intel|AMD)\s?([A-Za-z0-9\s]+)', name, re.IGNORECASE)
    if match:
        return f"{match.group(1)} {match.group(2).strip()}"
    return None

'''
def extract_cpu(title):
    if not isinstance(title, str):
        return None

    title = title.lower()

    if "i3" in title:
        return "i3"
    elif "i5" in title:
        return "i5"
    elif "i7" in title:
        return "i7"
    elif "ryzen 3" in title:
        return "Ryzen 3"
    elif "ryzen 5" in title:
        return "Ryzen 5"
    elif "ryzen 7" in title:
        return "Ryzen 7"

    return None


# Extracting (model) information from the 'name' column
brands = ["hp", "dell", "lenovo", "asus", "acer", "apple", 
          "ageneral", "duslang", "Foam Cleaner", "generic"]

def extract_model(name):
    if not isinstance(name, str):
        return None
    
    name_clean = re.sub(r'[^\w\s]', ' ', name.lower())
    words = name_clean.split()

    model_words = []
    found_brand = False

    for word in words:
        if word in brands:
            found_brand = True
        
        if not found_brand:
            continue
        
        if re.search(r'\d+gb|\d+tb|ram|ssd|hdd|intel|core', word):
            break
        
        model_words.append(word)

    if len(model_words) >= 2:
        return " ".join(model_words).title()
    
    return None;

<>:46: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:46: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\HP\AppData\Local\Temp\ipykernel_15156\3768216692.py:46: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  match = re.search(r'(Intel|AMD)\s?([A-Za-z0-9\s]+)', name, re.IGNORECASE)


In [32]:
data_jumia["ram_gb"] = data_jumia["name"].apply(extract_ram)
data_jumia["storage_gb"] = data_jumia["name"].apply(extract_storage)
data_jumia["storage_type"] = data_jumia["name"].apply(extract_storage_type)
data_jumia["cpu"] = data_jumia["name"].apply(extract_cpu)
data_jumia["model"] = data_jumia["name"].apply(extract_model)

data_jumia.head()

,name,price,old_price,discount,ram_gb,storage_gb,storage_type,cpu,model
0,Refurbished HP Probook 640 Core I5 8GB Ram 500...,559000.0,1400000.0,60%,8.0,8.0,HDD,i5,Hp Probook 640
1,Hp Refurbished Probook TouchScreen X360 Intel ...,440000.0,980019.0,55%,4.0,4.0,SSD,NaN,Hp Refurbished Probook Touchscreen X360
2,Latitude 11'' Inch 3180/3190 4GB RAM 128GB SSD...,349000.0,700000.0,50%,4.0,4.0,SSD,NaN,NaN
3,"11e Mini laptop,11.6"" Inch 4GB RAM,128GB SSD, ...",329000.0,800000.0,59%,4.0,4.0,SSD,NaN,NaN
4,"Travelmate 12 Inch Laptop 4GB RAM 128GB SSD, I...",356400.0,800000.0,55%,4.0,4.0,SSD,NaN,NaN


In [33]:
ram = data_jumia['ram_gb'].isna().sum()
storage = data_jumia['storage_gb'].isna().sum()
storage_type = data_jumia['storage_type'].isna().sum()
cpu = data_jumia['cpu'].isna().sum()
model = data_jumia['model'].isna().sum()


print(f"Missing RAM values: {ram}")
print(f"Missing Storage values: {storage}")
print(f"Missing Storage Type values: {storage_type}")
print(f"Missing CPU values: {cpu}")
print(f"Missing Model values: {model}")

Missing RAM values: 232
Missing Storage values: 46
Missing Storage Type values: 218
Missing CPU values: 674
Missing Model values: 1459
